# Text Signal & Success Analysis

Builds on the pillar scores by recomputing keyword densities, comparing grade-only vs. text-enhanced fidelity, and drilling into stratifications, residuals, and late-round prospects.


In [1]:
import warnings
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import nltk

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
pd.set_option('display.max_columns', 30)

for resource in ['punkt', 'stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(resource, quiet=True)


In [2]:
# Controls
START_YEAR = 2014
END_YEAR = 2025
TARGET_CUTOFF_YEAR = 2021
TARGET_COLUMN = 'made_it_contract'
LIMIT_TARGET_TO_CUTOFF = TARGET_COLUMN == 'made_it_contract'
LATE_ROUND_THRESHOLD = 64


In [3]:
CURATED_PHRASE_MAP = {
    'change of direction': 'change_of_direction',
    'low pad level': 'low_pad_level',
    'run after catch': 'run_after_catch',
    'yards after contact': 'yards_after_contact',
    'yards after catch': 'yards_after_catch',
    'off the line': 'off_the_line',
    'off the ball': 'off_the_ball',
    'point of attack': 'point_of_attack',
    'pass rush': 'pass_rush',
    'pass rusher': 'pass_rusher',
    'pass protection': 'pass_protection',
    'pass coverage': 'pass_coverage',
    'pad level': 'pad_level',
    'press coverage': 'press_coverage',
    'man coverage': 'man_coverage',
    'zone coverage': 'zone_coverage',
    'ball skills': 'ball_skills',
    'ball hawk': 'ball_hawk',
    'ball carrier': 'ball_carrier',
    'body control': 'body_control',
    'contact balance': 'contact_balance',
    'closing speed': 'closing_speed',
    'lateral quickness': 'lateral_quickness',
    'quick twitch': 'quick_twitch',
    'high motor': 'high_motor',
    'first step': 'first_step',
    'get off': 'get_off',
    'hand fighting': 'hand_fighting',
    'hand strength': 'hand_strength',
    'block shedding': 'block_shedding',
    'anchor strength': 'anchor_strength',
    'route running': 'route_running',
    'run blocking': 'run_blocking',
    'open field': 'open_field',
    'red zone': 'red_zone',
    'second level': 'second_level',
    'hip flexibility': 'hip_flexibility',
    'soft hands': 'soft_hands',
    'heavy hands': 'heavy_hands',
    'strong hands': 'strong_hands',
    'short area': 'short_area',
    'three down': 'three_down',
    'top end': 'top_end',
    'two gap': 'two_gap',
    'one gap': 'one_gap',
    'snap count': 'snap_count',
    'game ready': 'game_ready',
    'hard worker': 'hard_worker',
    'work ethic': 'work_ethic',
}

CURATED_PHRASE_MAP = dict(sorted(CURATED_PHRASE_MAP.items(), key=lambda x: len(x[0]), reverse=True))

KEEP_WORDS = {
    'high', 'low', 'heavy', 'light', 'deep', 'short', 'long', 'wide',
    'hard', 'soft', 'strong', 'quick', 'good', 'great', 'up', 'down',
    'off', 'out', 'over', 'through', 'above', 'below',
}

CUSTOM_STOPS = {
    'prospect', 'player', 'players', 'show', 'shows', 'need', 'needs',
    'ability', 'also', 'often', 'must', 'well', 'still', 'use', 'get',
    'make', 'look', 'help', 'time', 'year', 'team', 'game',
    'continue', 'develop', 'development', 'nfl', 'draft', 'college',
    'level', 'type', 'project', 'potential', 'upside', 'ceiling',
}

_base_stops = set(nltk.corpus.stopwords.words('english'))
NFL_STOPWORDS = (_base_stops - KEEP_WORDS) | CUSTOM_STOPS

lemmatizer = nltk.stem.WordNetLemmatizer()

def nfl_preprocess(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower()
    text = re.sub(r'[-\u2013\u2014]', ' ', text)
    for phrase, token in CURATED_PHRASE_MAP.items():
        text = text.replace(phrase, token)
    text = re.sub(r'[^a-z_\s]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if '_' in t or t not in NFL_STOPWORDS]
    tokens = [t if '_' in t else lemmatizer.lemmatize(t) for t in tokens]
    tokens = [t for t in tokens if len(t) > 1]
    return ' '.join(tokens)

PHYSICAL_KEYWORDS = {
    'explosive', 'burst', 'speed', 'quick_twitch', 'acceleration',
    'first_step', 'change_of_direction', 'agility', 'frame', 'size',
    'height', 'length', 'wingspan', 'natural', 'raw', 'powerful',
    'physical', 'vertical', 'leap', 'closing_speed', 'top_end',
    'big', 'tall', 'wide', 'lean', 'fast', 'athlete', 'elite', 'rare',
    'specimen', 'fluid', 'fluidity', 'projectable', 'strength', 'mass',
    'long', 'heavy', 'dynamic', 'shifty', 'explosion',
}

CHANGEABLE_KEYWORDS = {
    'technique', 'footwork', 'leverage', 'pad_level', 'hand_fighting',
    'anchor_strength', 'route_running', 'pass_protection', 'press_coverage',
    'zone_coverage', 'run_blocking', 'work_ethic', 'hard_worker', 'game_ready',
    'film_junkie', 'blue_collar', 'football_iq', 'shed', 'disrupt', 'compete',
    'hustle', 'grit', 'smart', 'cerebral', 'coachable', 'diagnose', 'read',
    'craft', 'savvy', 'consistent', 'reliable', 'dependable', 'finish', 'skill',
    'toughness', 'effort', 'relentless', 'motor', 'high_motor', 'intelligence',
    'awareness', 'recognition', 'instinct', 'discipline', 'fundamental',
    'mechanics', 'polish', 'refined', 'sound', 'precise', 'detail', 'process',
    'workhorse', 'stamina', 'energy', 'focus', 'intensity', 'coachability',
    'adapt', 'learn', 'adjust', 'improve',
}

def keyword_stats(text: str):
    tokens = text.split()
    total = len(tokens)
    physical = sum(tok in PHYSICAL_KEYWORDS for tok in tokens)
    changeable = sum(tok in CHANGEABLE_KEYWORDS for tok in tokens)
    return physical, changeable, total


In [4]:
# Load and filter data
raw_df = pd.read_csv('data/processed/draft_enriched_with_contracts.csv')
pillar_df = pd.read_csv('data/processed/pillar_scores_v3_2_bin.csv')

merged = raw_df.merge(
    pillar_df,
    on=['player_name', 'year', 'position', 'Pos_Group'],
    how='inner',
    suffixes=('_draft', '_pillar'),
)

for field in ['grade', TARGET_COLUMN, 'rank']:
    if field not in merged:
        for suffix in ['_draft', '_pillar']:
            candidate = f'{field}{suffix}'
            if candidate in merged:
                merged[field] = merged[candidate]
                break

merged = merged[merged['year'].between(START_YEAR, END_YEAR)]
if LIMIT_TARGET_TO_CUTOFF:
    merged = merged[merged['year'] <= TARGET_CUTOFF_YEAR]

merged = merged.dropna(subset=[TARGET_COLUMN, 'grade', 'strengths', 'weaknesses'])
merged[TARGET_COLUMN] = merged[TARGET_COLUMN].astype(bool).astype(int)
merged['grade'] = pd.to_numeric(merged['grade'], errors='coerce')
merged = merged.dropna(subset=['grade']).reset_index(drop=True)

merged['outside_top2'] = merged['rank'] > LATE_ROUND_THRESHOLD

print('Rows after filtering:', len(merged))
print('Positive rate (made it):', merged[TARGET_COLUMN].mean())
print('Year span:', merged['year'].min(), 'to', merged['year'].max())
print('Outside top-two rounds:', int(merged['outside_top2'].sum()), 'rows')


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/draft_enriched_with_contracts.csv'

In [ ]:
# Compute keyword densities
for section in ['strengths', 'weaknesses']:
    clean_col = f'{section}_clean'
    merged[clean_col] = merged[section].apply(nfl_preprocess)
    stats = merged[clean_col].apply(keyword_stats)
    stats_df = pd.DataFrame(
        stats.tolist(),
        columns=[f'{section}_physical_count', f'{section}_changeable_count', f'{section}_token_count'],
        index=merged.index,
    )
    merged = pd.concat([merged, stats_df], axis=1)
    denom = stats_df[f'{section}_token_count'].replace(0, np.nan)
    merged[f'{section}_physical_density'] = stats_df[f'{section}_physical_count'] / denom
    merged[f'{section}_changeable_density'] = stats_df[f'{section}_changeable_count'] / denom

merged['strengths_changeable_density'] = merged['strengths_changeable_density'].fillna(0)
merged['weaknesses_changeable_density'] = merged['weaknesses_changeable_density'].fillna(0)
merged['strengths_physical_density'] = merged['strengths_physical_density'].fillna(0)
merged['weaknesses_physical_density'] = merged['weaknesses_physical_density'].fillna(0)
merged['changeability_balance'] = (
    merged['strengths_changeable_density'] - merged['weaknesses_changeable_density']
)
merged['changeable_gap_strengths'] = (
    merged['strengths_changeable_density'] - merged['strengths_physical_density']
)
merged['changeable_gap_weaknesses'] = (
    merged['weaknesses_changeable_density'] - merged['weaknesses_physical_density']
)

print('Text feature snapshot (group means):')
print(merged.groupby(TARGET_COLUMN)[[
    'strengths_changeable_density',
    'strengths_physical_density',
    'weaknesses_changeable_density',
    'weaknesses_physical_density',
    'changeability_balance',
]].mean().round(3))


In [ ]:
# Modeling grade-only vs text-enhanced signals
feature_cols = [
    'grade',
    'score_god_given',
    'score_learned',
    'strengths_changeable_density',
    'strengths_physical_density',
    'weaknesses_changeable_density',
    'weaknesses_physical_density',
    'changeability_balance',
]

X = merged[feature_cols]
y = merged[TARGET_COLUMN]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)
train_idx, test_idx = X_train.index, X_test.index

scaler_all = StandardScaler()
X_train_scaled = scaler_all.fit_transform(X_train)
X_test_scaled = scaler_all.transform(X_test)
clf_text = LogisticRegression(max_iter=1000)
clf_text.fit(X_train_scaled, y_train)

grade_scaler = StandardScaler()
X_grade = merged[['grade']]
Xg_train = grade_scaler.fit_transform(X_grade.loc[train_idx])
Xg_test = grade_scaler.transform(X_grade.loc[test_idx])
clf_grade = LogisticRegression(max_iter=1000)
clf_grade.fit(Xg_train, y_train)

print('Grade-only accuracy:', clf_grade.score(Xg_test, y_test))
print('Grade + text accuracy:', clf_text.score(X_test_scaled, y_test))
print('\nText-enhanced classification report:')
print(classification_report(y_test, clf_text.predict(X_test_scaled), digits=3))

coef_df = pd.DataFrame({'feature': feature_cols, 'coef': clf_text.coef_[0]})
coef_df = coef_df.sort_values('coef', ascending=False)
sns.barplot(data=coef_df, x='coef', y='feature', palette='viridis')
plt.title('Logistic coefficients (text-enhanced model)')
plt.tight_layout()
coef_df


In [ ]:
# Stratified modeling by Pos_Group
group_metrics = []
for group, subset in merged.groupby('Pos_Group'):
    if len(subset) < 40:
        continue
    Xg = subset[feature_cols]
    yg = subset[TARGET_COLUMN]
    try:
        Xg_train, Xg_test, yg_train, yg_test = train_test_split(
            Xg, yg, stratify=yg, test_size=0.2, random_state=42
        )
    except ValueError:
        continue

    scaler_grp = StandardScaler()
    Xg_train_scaled = scaler_grp.fit_transform(Xg_train)
    Xg_test_scaled = scaler_grp.transform(Xg_test)

    clf_grp_text = LogisticRegression(max_iter=1000)
    clf_grp_text.fit(Xg_train_scaled, yg_train)

    scaler_grp_grade = StandardScaler()
    Xg_grade = subset[['grade']]
    Xg_grade_train = scaler_grp_grade.fit_transform(Xg_grade.loc[Xg_train.index])
    Xg_grade_test = scaler_grp_grade.transform(Xg_grade.loc[Xg_test.index])
    clf_grp_grade = LogisticRegression(max_iter=1000)
    clf_grp_grade.fit(Xg_grade_train, yg_train)

    group_metrics.append({
        'Pos_Group': group,
        'n': len(subset),
        'grade_acc': clf_grp_grade.score(Xg_grade_test, yg_test),
        'text_acc': clf_grp_text.score(Xg_test_scaled, yg_test),
        'gain': clf_grp_text.score(Xg_test_scaled, yg_test) - clf_grp_grade.score(Xg_grade_test, yg_test),
    })

group_metrics_df = pd.DataFrame(group_metrics).sort_values('gain', ascending=False)
group_metrics_df


In [ ]:
# Residual analysis (text-enhanced model)
test_probs_text = clf_text.predict_proba(X_test_scaled)[:, 1]
test_probs_grade = clf_grade.predict_proba(Xg_test)[:, 1]
residuals = y_test - test_probs_text

resid_df = pd.DataFrame({
    'residual': residuals,
    'prob_text': test_probs_text,
    'grade': X_test['grade'],
    'score_learned': X_test['score_learned'],
    'score_god_given': X_test['score_god_given'],
}, index=X_test.index)
resid_df['Pos_Group'] = merged.loc[X_test.index, 'Pos_Group']

sns.histplot(resid_df['residual'], kde=True, bins=20)
plt.title('Residuals (actual - prob_text)')
plt.show()

sns.scatterplot(data=resid_df, x='grade', y='residual', hue='Pos_Group', alpha=0.55)
plt.title('Residual vs. grade')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

print('Residual balance by Pos_Group:')
print(resid_df.groupby('Pos_Group')['residual'].mean().round(3))


In [ ]:
# Late-round (outside top 2) performance
late_df = merged[merged['outside_top2']]
print('Late-round rows:', len(late_df))
if len(late_df) >= 100:
    X_late = late_df[feature_cols]
    y_late = late_df[TARGET_COLUMN]
    Xl_train, Xl_test, yl_train, yl_test = train_test_split(
        X_late, y_late, stratify=y_late, test_size=0.2, random_state=42
    )
    scaler_late = StandardScaler()
    Xl_train_scaled = scaler_late.fit_transform(Xl_train)
    Xl_test_scaled = scaler_late.transform(Xl_test)

    clf_late = LogisticRegression(max_iter=1000)
    clf_late.fit(Xl_train_scaled, yl_train)

    print('Late-round text-enhanced accuracy:', clf_late.score(Xl_test_scaled, yl_test))
    print('Classification report (late-round):')
    print(classification_report(yl_test, clf_late.predict(Xl_test_scaled), digits=3))
else:
    print('Not enough late-round rows to train a dedicated model.')


In [ ]:
# Highlight rescued players for the full model
test_df = merged.loc[test_idx, ['player_name', 'year', 'position', 'Pos_Group', 'grade', TARGET_COLUMN]].copy()
test_df['prob_grade_only'] = test_probs_grade
test_df['prob_text'] = test_probs_text
test_df['prob_diff'] = test_df['prob_text'] - test_df['prob_grade_only']
test_df['grade_only_pred'] = clf_grade.predict(Xg_test)
test_df['text_pred'] = clf_text.predict(X_test_scaled)
rescued = test_df[
    (test_df[TARGET_COLUMN] == 1) &
    (test_df['grade_only_pred'] == 0) &
    (test_df['text_pred'] == 1)
].sort_values('prob_diff', ascending=False).head(5)
print('Top players where text flipped the prediction:')
rescued


## Summary
- Stratifying by `Pos_Group` shows how much the text-enhanced model helps each positional cohort; keep an eye on the `gain` column for the biggest improvements.
- Residual plots reveal where the text model still misses: mostly on the heavier grade end, but some positions are systematically over- or under-predicted.
- The late-round model isolates prospects outside the top 64 ranked players, showing whether the language signal can identify sleeper-made-it cases that grade alone might miss.
